# Small LM analysis

Loads eval results for a trained run and plots them. wandb is optional — this
notebook reads metrics from disk (`source="local"`), so it works with wandb off.

## Setup

In [ ]:
import os, glob, sys
from pyprojroot import here
sys.path.append(str(here()))
%matplotlib inline
import torch
import matplotlib.pyplot as plt
from src.code.analysis_utils import load_eval_results

## Load eval results (offline)

One row per (run, checkpoint step), with a `config` column and flat metric
columns such as `loss`. Pass `source="wandb"` to pull the same shape from wandb.

In [ ]:
setting = "balls-and-urns"
exp_name = "small-lm"  # the name set by runner=train_small_lm

df = load_eval_results(setting, exp_name).sort_values("step")
df.head()

## Eval loss across checkpoints

The Bayesian baseline (`runner=eval_bayes`) is a single run, so we draw it as a
reference line.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(df["step"], df["loss"], marker="o", label="small LM")

try:
    bayes_loss = load_eval_results(setting, "bayes-eval").iloc[0]["loss"]
    ax.axhline(bayes_loss, color="gray", ls="--", label="Bayesian baseline")
except (FileNotFoundError, IndexError):
    pass  # no Bayesian run available

ax.set_xlabel("training step")
ax.set_ylabel("eval loss (KL)")
ax.legend()
ax.grid(True)

## Per-position loss for one checkpoint

Per-example, per-position losses are saved under `extras` in each
`metrics.pt` (not loaded into the DataFrame). Load one directly to inspect how
loss varies across the context.

In [ ]:
cache_dir = os.getenv("CACHE_DIR", "./outputs")
pattern = os.path.join(cache_dir, setting, exp_name, "*", "checkpoints", "*", "metrics.pt")
latest = sorted(glob.glob(pattern))[-1]

all_loss = torch.as_tensor(torch.load(latest, weights_only=False)["extras"]["all_loss"])
per_position = all_loss.mean(dim=0)  # mean over examples -> (context_length,)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(range(1, len(per_position) + 1), per_position, marker=".")
ax.set_xlabel("context position")
ax.set_ylabel("mean KL")
ax.grid(True)